# Task 2: Premium Sales Performance Dashboard

**Optimus Automate Data Analytics Internship 2026**

This notebook builds a professional executive dashboard using the enhanced Sample Superstore dataset. It creates KPI cards, detailed charts, business insights, and exports one complete HTML dashboard file.


## 1. Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

pd.set_option('display.max_columns', None)


## 2. Load dataset

In [2]:
DATA_PATH = '../data/SampleSuperstore.csv'
df = pd.read_csv(DATA_PATH)
df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Order Priority,Customer ID,Customer Name,Customer Tenure Group,Customer Age Group,Segment,Country,City,State,Postal Code,Region,Sales Channel,Payment Method,Returned,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2025-100000,2025-05-10,2025-05-19,Second Class,Critical,CUST-6141,Zain Siddiqui,New,26-35,Home Office,United States,Seattle,Washington,98101.0,West,Online,Credit Card,No,TEC-PR-3005,Technology,Printers,Inkjet Printer,1199.07,8,0.15,-91.87
1,2,CA-2025-100001,2025-11-10,2025-11-15,Standard Class,Medium,CUST-9501,Ayesha Ahmed,3-5 Years,26-35,Consumer,United States,Detroit,Michigan,48201.0,Central,Online,PayPal,No,TEC-PH-3001,Technology,Phones,Smart Office Phone,572.25,4,0.20,75.54
2,3,CA-2023-100002,2023-05-02,2023-05-09,Second Class,Medium,CUST-3708,Ahmed Butt,1-2 Years,26-35,Home Office,United States,Detroit,Michigan,48201.0,Central,Retail Store,Credit Card,No,TEC-PR-3005,Technology,Printers,Inkjet Printer,3099.41,7,0.05,647.97
3,4,CA-2024-100003,2024-04-11,2024-04-15,Second Class,Medium,CUST-2991,Noor Iqbal,New,18-25,Home Office,United States,Minneapolis,Minnesota,55401.0,Central,Retail Store,Credit Card,No,TEC-AC-3002,Technology,Accessories,Wireless Mouse,221.01,3,0.00,27.09
4,5,CA-2023-100004,2023-11-27,2023-12-01,Standard Class,Low,CUST-9463,Faraha Khan,New,26-35,Corporate,United States,Nashville,Tennessee,37201.0,South,Partner Reseller,Bank Transfer,No,FUR-TA-1002,Furniture,Tables,Executive Conference Table,1389.85,5,0.10,31.46


## 3. Data preparation and feature engineering

In [3]:
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Ensure core calculated fields exist
df['Order Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['Order Year'] = df['Order Date'].dt.year
df['Profit Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], 0)
df['Discount Band'] = pd.cut(df['Discount'], bins=[-0.001, 0, 0.1, 0.2, 0.5, 1.0], labels=['No discount', 'Low', 'Medium', 'High', 'Very high'])
df['Loss Flag'] = np.where(df['Profit'] < 0, 'Loss', 'Profit')
df['Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1505 entries, 0 to 1504
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Row ID                 1505 non-null   int64         
 1   Order ID               1505 non-null   str           
 2   Order Date             1505 non-null   datetime64[us]
 3   Ship Date              1505 non-null   datetime64[us]
 4   Ship Mode              1505 non-null   str           
 5   Order Priority         1505 non-null   str           
 6   Customer ID            1505 non-null   str           
 7   Customer Name          1505 non-null   str           
 8   Customer Tenure Group  1505 non-null   str           
 9   Customer Age Group     1505 non-null   str           
 10  Segment                1505 non-null   str           
 11  Country                1505 non-null   str           
 12  City                   1497 non-null   str           
 13  State         

## 4. KPI calculations

In [4]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin = total_profit / total_sales if total_sales else 0
orders = df['Order ID'].nunique()
customers = df['Customer ID'].nunique()
aov = total_sales / orders if orders else 0
qty = df['Quantity'].sum()
avg_discount = df['Discount'].mean()
loss_orders = df[df['Profit'] < 0]['Order ID'].nunique()
loss_rate = loss_orders / orders if orders else 0
avg_ship_days = df['Processing Days'].mean()

kpi_summary = pd.DataFrame({
    'Metric': ['Total Sales','Total Profit','Profit Margin','Total Orders','Customers','Average Order Value','Units Sold','Loss Order Rate'],
    'Value': [total_sales,total_profit,profit_margin,orders,customers,aov,qty,loss_rate]
})
kpi_summary


,Metric,Value
0,Total Sales,1.304682e+06
1,Total Profit,1.132957e+05
2,Profit Margin,8.683780e-02
3,Total Orders,1.500000e+03
4,Customers,1.373000e+03
5,Average Order Value,8.697878e+02
6,Units Sold,6.085000e+03
7,Loss Order Rate,2.480000e-01


## 5. Aggregated dashboard data

In [5]:
monthly = df.groupby('Order Month', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'))
regional = df.groupby('Region', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique')).sort_values('Sales', ascending=False)
category = df.groupby('Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique')).sort_values('Sales', ascending=False)
subcategory = df.groupby('Sub-Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales', ascending=False).head(12)
segment = df.groupby('Segment', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Customers=('Customer ID','nunique')).sort_values('Sales', ascending=False)
ship = df.groupby('Ship Mode', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'), Avg_Ship_Days=('Processing Days','mean')).sort_values('Sales', ascending=False)
top_products = df.groupby('Product Name', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique')).sort_values('Sales', ascending=False).head(10)
state = df.groupby('State', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales', ascending=False).head(10)
discount = df.groupby('Discount Band', observed=True, as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique')).sort_values('Discount Band')

display(monthly.head())
display(regional)
display(category)


,Order Month,Sales,Profit,Orders
0,2023-01,39860.89,3700.00,61
1,2023-02,25615.52,1640.39,42
2,2023-03,44202.33,7009.23,38
3,2023-04,32875.02,4764.79,47
4,2023-05,47635.27,3549.49,44


,Region,Sales,Profit,Orders
3,West,355546.20,30657.67,388
2,South,331513.30,30031.32,345
0,Central,309474.22,28646.81,357
1,East,308147.97,23959.89,410


,Category,Sales,Profit,Orders
2,Technology,815318.27,102246.32,566
0,Furniture,446418.75,8303.21,435
1,Office Supplies,42944.67,2746.16,499


## 6. Create interactive Plotly charts

In [6]:
template = 'plotly_dark'
plotly_config = {'displayModeBar': True, 'responsive': True}

fig_monthly = make_subplots(specs=[[{'secondary_y': False}]])
fig_monthly.add_trace(go.Bar(x=monthly['Order Month'], y=monthly['Sales'], name='Sales'))
fig_monthly.add_trace(go.Scatter(x=monthly['Order Month'], y=monthly['Profit'], name='Profit', mode='lines+markers'))
fig_monthly.update_layout(template=template, title='Monthly Sales and Profit Trend', height=430, legend=dict(orientation='h', y=-0.2), margin=dict(l=30,r=30,t=70,b=70))
fig_monthly.update_yaxes(title='Amount')

fig_region = px.bar(regional, x='Region', y=['Sales','Profit'], barmode='group', title='Regional Performance: Sales vs Profit', template=template, height=430)
fig_cat = px.pie(category, names='Category', values='Sales', title='Sales Mix by Category', template=template, hole=0.52, height=430)
fig_products = px.bar(top_products.sort_values('Sales'), y='Product Name', x='Sales', orientation='h', title='Top 10 Products by Sales', template=template, height=520, hover_data=['Profit','Orders'])
fig_sub = px.bar(subcategory.sort_values('Sales'), y='Sub-Category', x=['Sales','Profit'], orientation='h', barmode='group', title='Top Sub-Categories: Sales and Profit', template=template, height=500)
fig_segment = px.bar(segment, x='Segment', y=['Sales','Profit'], barmode='group', title='Customer Segment Performance', template=template, height=430, hover_data=['Customers'])
fig_ship = px.bar(ship, x='Ship Mode', y='Orders', color='Profit', title='Shipping Mode Performance by Orders and Profit', template=template, height=430, hover_data=['Sales','Avg_Ship_Days'])
fig_state = px.bar(state.sort_values('Sales'), y='State', x='Sales', orientation='h', title='Top 10 States by Sales', template=template, height=500, hover_data=['Profit'])
fig_discount = px.bar(discount, x='Discount Band', y=['Sales','Profit'], barmode='group', title='Discount Impact on Sales and Profit', template=template, height=430)
fig_scatter = px.scatter(df, x='Sales', y='Profit', color='Category', size='Quantity', hover_data=['Product Name','Region','Segment'], title='Sales vs Profit Relationship by Category', template=template, height=500)

fig_monthly.show()


## 7. Export one premium HTML dashboard

In [7]:
# The complete HTML dashboard is generated by the companion file provided with this package.
# To export from this notebook, use the ready-made HTML file or rerun the Python generation script in README_TASK2_PREMIUM.txt.
os.makedirs('../dashboard', exist_ok=True)
for name, fig in {
    'monthly_sales_profit_trend': fig_monthly,
    'regional_performance': fig_region,
    'category_sales_mix': fig_cat,
    'top_products': fig_products,
    'subcategory_performance': fig_sub,
    'segment_performance': fig_segment,
    'shipping_performance': fig_ship,
    'state_performance': fig_state,
    'discount_impact': fig_discount,
    'sales_profit_scatter': fig_scatter,
}.items():
    fig.write_html(f'../dashboard/{name}.html')

print('Individual chart HTML files exported to the dashboard folder.')


Individual chart HTML files exported to the dashboard folder.


## 8. Business recommendations

1. Protect high-performing regions with stronger stock planning and campaign support.
2. Track profit together with sales, because high sales can still hide weak margins.
3. Review discount-heavy orders and categories that produce negative profit.
4. Use top product analysis for inventory prioritization.
5. Use shipping mode analysis to identify operational efficiency opportunities.
